# Classifying NDVI into Land Cover Groups

**Series:** NDVI and Remote Sensing (notebook 2 of 3)
**Time:** about 20 minutes

**In this notebook you will:**
- Group NDVI values into classes with simple thresholds
- Make a categorical map with a clear legend
- Measure how much of the scene is in each class
- Mask out water

**You should already know:** notebook 05a (computing NDVI).

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## From numbers to categories

A continuous NDVI map is hard to summarize. Grouping pixels into a few classes lets us say things like
"40% of this area is dense vegetation."

## Step 1: Download, load and compute NDVI (same as notebook 05a)

In [ ]:
!pip install -q rasterio

import numpy as np
import matplotlib.pyplot as plt
import rasterio

# rasterio reads the bands straight from GitHub into memory; nothing is downloaded to disk
RED_URL = "https://raw.githubusercontent.com/SpatialTurn/DataAnalysis/main/data/landsat/band4.TIF"   # Band 4: red
NIR_URL = "https://raw.githubusercontent.com/SpatialTurn/DataAnalysis/main/data/landsat/band5.TIF"   # Band 5: near infrared (NIR)

In [ ]:
with rasterio.open(RED_URL) as src:
    red = src.read(1).astype("float32")
    profile = src.profile          # size, CRS and position of the image
with rasterio.open(NIR_URL) as src:
    nir = src.read(1).astype("float32")

# 0 marks the empty corners outside the satellite's view: treat it as missing
red[red == 0] = np.nan
nir[nir == 0] = np.nan

# Convert stored values to surface reflectance (Landsat Collection 2 Level-2 scale factors)
red = np.clip(red * 0.0000275 - 0.2, 0, 1)
nir = np.clip(nir * 0.0000275 - 0.2, 0, 1)

ndvi = (nir - red) / (nir + red)
del red, nir                      # free memory: we only need NDVI from here on

## Step 2: Choose class breaks

We use four classes. `np.digitize` returns the class number for each pixel:

| Class | NDVI range | Label |
|---|---|---|
| 0 | below 0 | Water |
| 1 | 0 to 0.2 | Bare or built up |
| 2 | 0.2 to 0.4 | Sparse vegetation |
| 3 | 0.4 and above | Dense vegetation |

Four classes need **three** break values.

In [ ]:
breaks = [0, 0.2, 0.4]
classes = np.digitize(ndvi, breaks).astype("float32")
classes[np.isnan(ndvi)] = np.nan          # keep the empty edges empty

names = ["Water", "Bare or built up", "Sparse vegetation", "Dense vegetation"]
colors = ["#2b83ba", "#d7a26b", "#a6d96a", "#1a9641"]

## Step 3: Map the classes

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

plt.figure(figsize=(9, 8))
# Show every 4th pixel: the screen cannot display 60 million pixels anyway,
# and drawing all of them can run Colab out of memory
plt.imshow(classes[::4, ::4], cmap=ListedColormap(colors), vmin=0, vmax=3, interpolation="nearest")
plt.legend(handles=[Patch(color=c, label=n) for c, n in zip(colors, names)],
           loc="lower left", fontsize=9)
plt.title("NDVI classes")
plt.axis("off")
plt.show()

There is one color per class, and the legend matches the table above.

## Step 4: How much of the scene is in each class?

Each Landsat pixel is 30 m by 30 m, which is 0.0009 square kilometers.

In [ ]:
import pandas as pd

valid = classes[~np.isnan(classes)].astype(int)
counts = np.bincount(valid, minlength=4)
summary = pd.DataFrame({
    "class": names,
    "pixels": counts,
    "area_km2": (counts * 0.0009).round(0),
    "percent": (100 * counts / counts.sum()).round(1),
})
summary

## Step 5: Mask water

Sometimes we want to ignore water entirely, for example when averaging vegetation health.
`np.where` keeps NDVI where the condition is false and puts `NaN` where it is true.

In [ ]:
ndvi_land = np.where(ndvi < 0, np.nan, ndvi)

plt.figure(figsize=(9, 8))
plt.imshow(ndvi_land[::4, ::4], cmap="YlGn", vmin=0, vmax=0.8)
plt.colorbar(label="NDVI (land only)")
plt.title("NDVI with water removed")
plt.axis("off")
plt.show()

print("Mean NDVI, land only:", round(float(np.nanmean(ndvi_land)), 3))

> **Thresholds depend on your data.** These breaks are a common starting point, not a rule.
> Season matters most: a corn field has a much lower NDVI after harvest than in July, so check the
> histogram from notebook 05a and adjust the breaks for your scene.

## Try it yourself

Try **five** classes with breaks `[0, 0.1, 0.2, 0.3]`. Update `names` and `colors` to match, then redraw the map and summary table.

*Hint:* n breaks make n + 1 classes, so you need 5 names and 5 colors.

<details>
<summary><b>Show a solution</b></summary>

```python
breaks = [0, 0.1, 0.2, 0.3]
names = ["Water", "Bare", "Built up or very sparse", "Sparse vegetation", "Dense vegetation"]
colors = ["#2b83ba", "#d7a26b", "#fdae61", "#a6d96a", "#1a9641"]
classes = np.digitize(ndvi, breaks).astype("float32")
classes[np.isnan(ndvi)] = np.nan
plt.imshow(classes[::4, ::4], cmap=ListedColormap(colors), vmin=0, vmax=4, interpolation="nearest")
plt.show()
```

</details>

In [ ]:
# Write your answer here

## Recap

- `np.digitize` turns continuous values into class numbers; n breaks give n + 1 classes.
- Use one color per class and a legend for categorical maps.
- Pixel counts times pixel area give the area of each class.
- `np.where` masks unwanted pixels such as water.

**Next:** [Image Classification (advanced)](https://colab.research.google.com/github/SpatialTurn/DataAnalysis/blob/main/notebooks/05c_Image_Classification_Advanced.ipynb)